# Interactive River Network Map

This notebook creates a polished interactive Folium map of your compressed river network.

Toggleable layers include:
- training 5/5 WQ stations
- held-out 5/5 test stations
- partial WQ test stations
- hydrology support gauges
- network sources / upstream leaves
- downstream outlets / leaves
- junctions / meeting points
- the river network itself

Every marker has a popup with graph information and a direct **Google Maps** link. The map also includes fullscreen mode, a minimap, mouse coordinates, distance measuring, and a search box.

### Files expected in the same folder

- `master_compressed_river_graph.pkl`
- `largest_river_component.pkl` *(optional, but recommended for accurate curved river paths)*

If the detailed graph is missing, the notebook still works; compressed edges are just drawn straight between retained nodes.

In [3]:
import os
import pickle
import html
import pandas as pd
import numpy as np
import networkx as nx
import folium
from folium import plugins
from branca.element import Element

MASTER_GRAPH_FILE = "master_compressed_river_graph.pkl"
DETAILED_GRAPH_FILE = "largest_river_component.pkl"

OUTPUT_HTML = "interactive_river_network_map.html"
OUTPUT_POINTS_CSV = "interactive_river_map_points.csv"

print("Working directory:", os.getcwd())
print("Master graph exists:", os.path.exists(MASTER_GRAPH_FILE))
print("Detailed graph exists:", os.path.exists(DETAILED_GRAPH_FILE))

Working directory: /Users/andrewchincea/TDA_Lab_Practice/rivernets/making_the_graph
Master graph exists: True
Detailed graph exists: True


## 1. Load the graphs

In [4]:
if not os.path.exists(MASTER_GRAPH_FILE):
    raise FileNotFoundError(
        f"{MASTER_GRAPH_FILE} was not found in {os.getcwd()}"
    )

with open(MASTER_GRAPH_FILE, "rb") as f:
    G = pickle.load(f)

if os.path.exists(DETAILED_GRAPH_FILE):
    with open(DETAILED_GRAPH_FILE, "rb") as f:
        G_detail = pickle.load(f)
    print("Detailed river geometry: AVAILABLE")
else:
    G_detail = None
    print("Detailed river geometry: NOT FOUND — using straight compressed edges")

print("\nCompressed graph")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Directed:", G.is_directed())

if G.is_directed():
    print("Weak components:", nx.number_weakly_connected_components(G))

Detailed river geometry: AVAILABLE

Compressed graph
Nodes: 67
Edges: 66
Directed: True
Weak components: 1


## 2. Classify structural nodes and monitoring roles

In [5]:
source_nodes = set()
outlet_nodes = set()
junction_nodes = set()

train_nodes = set()
full_test_nodes = set()
partial_test_nodes = set()
support_nodes = set()

for node, data in G.nodes(data=True):
    indeg = G.in_degree(node)
    outdeg = G.out_degree(node)

    if indeg == 0 and outdeg > 0:
        source_nodes.add(node)

    if indeg > 0 and outdeg == 0:
        outlet_nodes.add(node)

    if indeg + outdeg >= 3:
        junction_nodes.add(node)

    for cluster in data.get("usgs_clusters", []):
        role = cluster.get("ml_role")
        if role == "train":
            train_nodes.add(node)
        elif role == "test_full":
            full_test_nodes.add(node)
        elif role == "test_partial":
            partial_test_nodes.add(node)

    if data.get("support_gauge_count", 0) > 0:
        support_nodes.add(node)

print("Network sources:", len(source_nodes))
print("Outlets / downstream leaves:", len(outlet_nodes))
print("Junctions / meeting points:", len(junction_nodes))
print()
print("Training 5/5 locations:", len(train_nodes))
print("Held-out 5/5 test locations:", len(full_test_nodes))
print("Partial WQ test locations:", len(partial_test_nodes))
print("Hydrology support nodes:", len(support_nodes))

Network sources: 19
Outlets / downstream leaves: 1
Junctions / meeting points: 18

Training 5/5 locations: 10
Held-out 5/5 test locations: 4
Partial WQ test locations: 12
Hydrology support nodes: 3


## 3. Measure downstream source coverage

In [6]:
measurement_nodes = {
    node
    for node, data in G.nodes(data=True)
    if (
        data.get("usgs_station_count", 0) > 0
        or data.get("support_gauge_count", 0) > 0
    )
}

source_coverage_rows = []

for source in source_nodes:
    lengths = nx.single_source_dijkstra_path_length(
        G,
        source,
        weight="length_m"
    )

    reachable = {
        node: dist
        for node, dist in lengths.items()
        if node in measurement_nodes
    }

    if reachable:
        nearest = min(reachable, key=reachable.get)
        distance_km = reachable[nearest] / 1000
    else:
        nearest = None
        distance_km = np.nan

    source_coverage_rows.append({
        "source_node": source,
        "nearest_measurement_node": nearest,
        "distance_to_first_measurement_km": distance_km
    })

source_coverage = pd.DataFrame(source_coverage_rows)

if len(source_coverage):
    display(
        source_coverage.sort_values(
            "distance_to_first_measurement_km",
            ascending=False,
            na_position="first"
        )
    )

    print()
    print(
        source_coverage["distance_to_first_measurement_km"]
        .describe()
    )

,source_node,nearest_measurement_node,distance_to_first_measurement_km
5,188990639,NaN,NaN
11,209256258,NaN,NaN
13,202613460,NaN,NaN
16,204021741,NaN,NaN
0,128398218,203177183.0,729.734489
18,171485821,151170041.0,711.542365
10,132116925,203177183.0,673.670218
1,105675404,131520862.0,370.704374
12,177526604,203177183.0,328.067555
14,64455269,67739434.0,159.991380



count     15.000000
mean     243.585014
std      260.233127
min        1.910262
25%       70.286115
50%      142.879136
75%      349.385965
max      729.734489
Name: distance_to_first_measurement_km, dtype: float64


## 4. Build a searchable point table

In [7]:
source_distance_lookup = (
    source_coverage
    .set_index("source_node")["distance_to_first_measurement_km"]
    .to_dict()
    if len(source_coverage)
    else {}
)


def coords(node):
    data = G.nodes[node]
    return float(data["latitude"]), float(data["longitude"])


point_rows = []


def add_point(node, point_type, label, extra=None):
    lat, lon = coords(node)
    row = {
        "node_id": node,
        "type": point_type,
        "label": label,
        "latitude": lat,
        "longitude": lon,
    }
    if extra:
        row.update(extra)
    point_rows.append(row)


for node in sorted(source_nodes):
    distance = source_distance_lookup.get(node, np.nan)
    coverage = (
        "No downstream measurement"
        if pd.isna(distance)
        else f"{distance:.1f} km to first downstream measurement"
    )
    add_point(node, "Network source", f"Source node {node}", {"coverage": coverage})

for node in sorted(outlet_nodes):
    add_point(node, "Outlet / downstream leaf", f"Outlet node {node}")

for node in sorted(junction_nodes):
    add_point(
        node,
        "Junction / meeting point",
        f"Junction node {node}",
        {"in_degree": G.in_degree(node), "out_degree": G.out_degree(node)}
    )

for node in sorted(train_nodes):
    clusters = [c for c in G.nodes[node].get("usgs_clusters", []) if c.get("ml_role") == "train"]
    members = sorted({sid for c in clusters for sid in c.get("member_stations", [])})
    add_point(node, "Training — 5/5", " / ".join(members) if members else f"Training node {node}", {"member_stations": ", ".join(members)})

for node in sorted(full_test_nodes):
    clusters = [c for c in G.nodes[node].get("usgs_clusters", []) if c.get("ml_role") == "test_full"]
    members = sorted({sid for c in clusters for sid in c.get("member_stations", [])})
    add_point(node, "Test — held-out 5/5", " / ".join(members) if members else f"Full test node {node}", {"member_stations": ", ".join(members)})

for node in sorted(partial_test_nodes):
    clusters = [c for c in G.nodes[node].get("usgs_clusters", []) if c.get("ml_role") == "test_partial"]
    members = sorted({sid for c in clusters for sid in c.get("member_stations", [])})
    feature_counts = [c.get("feature_count") for c in clusters if c.get("feature_count") is not None]
    add_point(
        node,
        "Test — partial WQ",
        " / ".join(members) if members else f"Partial test node {node}",
        {"member_stations": ", ".join(members), "feature_count": max(feature_counts) if feature_counts else None}
    )

for node in sorted(support_nodes):
    gauges = G.nodes[node].get("support_gauges", [])
    gauge_ids = sorted({str(g.get("station_id")) for g in gauges if g.get("station_id") is not None})
    add_point(node, "Hydrology support", " / ".join(gauge_ids) if gauge_ids else f"Support node {node}", {"support_gauges": ", ".join(gauge_ids)})

points_df = pd.DataFrame(point_rows)
points_df.to_csv(OUTPUT_POINTS_CSV, index=False)

print("Saved:", OUTPUT_POINTS_CSV)
print()
print(points_df["type"].value_counts())

Saved: interactive_river_map_points.csv

type
Network source              19
Junction / meeting point    18
Test — partial WQ           12
Training — 5/5              10
Test — held-out 5/5          4
Hydrology support            3
Outlet / downstream leaf     1
Name: count, dtype: int64


## 5. Create the interactive map

In [11]:
from getpass import getpass
import html
import folium
import numpy as np
import pandas as pd

from folium import plugins
from branca.element import Element


# ==================================================
# CARTO BASEMAP API KEY
# ==================================================

CARTO_API_KEY = getpass(
    "Enter your CARTO Basemaps API key: "
).strip()

if not CARTO_API_KEY:
    raise ValueError(
        "A CARTO Basemaps API key is required."
    )


# ==================================================
# MAP CENTER AND BOUNDS
# ==================================================

all_lat = [
    float(data["latitude"])
    for _, data in G.nodes(data=True)
]

all_lon = [
    float(data["longitude"])
    for _, data in G.nodes(data=True)
]

center = [
    float(np.mean(all_lat)),
    float(np.mean(all_lon))
]


# ==================================================
# CREATE MAP
# ==================================================

m = folium.Map(
    location=center,
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True
)


# ==================================================
# BASE MAPS
# ==================================================

# -----------------------------
# OpenStreetMap
# -----------------------------

folium.TileLayer(
    tiles=(
        "https://tile.openstreetmap.org/"
        "{z}/{x}/{y}.png"
    ),
    name="OpenStreetMap",
    attr=(
        '&copy; '
        '<a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap"
        "</a> contributors"
    ),
    max_zoom=19,
    control=True,
    show=True
).add_to(m)


# -----------------------------
# CARTO Positron
# -----------------------------

folium.TileLayer(
    tiles=(
        "https://basemaps.cartocdn.com/"
        "rastertiles/light_all/"
        "{z}/{x}/{y}{r}.png"
        f"?key={CARTO_API_KEY}"
    ),
    name="CARTO Positron",
    attr=(
        '&copy; '
        '<a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap"
        "</a> contributors "
        '&copy; '
        '<a href="https://carto.com/attribution/">'
        "CARTO"
        "</a>"
    ),
    max_zoom=20,
    control=True,
    show=False
).add_to(m)


# -----------------------------
# CARTO Dark Matter
# -----------------------------

folium.TileLayer(
    tiles=(
        "https://basemaps.cartocdn.com/"
        "rastertiles/dark_all/"
        "{z}/{x}/{y}{r}.png"
        f"?key={CARTO_API_KEY}"
    ),
    name="CARTO Dark Matter",
    attr=(
        '&copy; '
        '<a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap"
        "</a> contributors "
        '&copy; '
        '<a href="https://carto.com/attribution/">'
        "CARTO"
        "</a>"
    ),
    max_zoom=20,
    control=True,
    show=False
).add_to(m)


# ==================================================
# FEATURE GROUPS
# ==================================================

river_layer = folium.FeatureGroup(
    name="River network",
    show=True
).add_to(m)

source_layer = folium.FeatureGroup(
    name="Network sources",
    show=True
).add_to(m)

outlet_layer = folium.FeatureGroup(
    name="Outlets / downstream leaves",
    show=True
).add_to(m)

junction_layer = folium.FeatureGroup(
    name="Junctions / meeting points",
    show=True
).add_to(m)

train_layer = folium.FeatureGroup(
    name="Training — 5/5",
    show=True
).add_to(m)

full_test_layer = folium.FeatureGroup(
    name="Test — held-out 5/5",
    show=True
).add_to(m)

partial_test_layer = folium.FeatureGroup(
    name="Test — partial WQ",
    show=True
).add_to(m)

support_layer = folium.FeatureGroup(
    name="Hydrology support",
    show=True
).add_to(m)


# ==================================================
# POPUP HELPERS
# ==================================================

def google_maps_url(lat, lon):

    return (
        "https://www.google.com/maps/search/"
        "?api=1"
        f"&query={lat},{lon}"
    )


def popup_html(
    node,
    title,
    details=None
):

    lat, lon = coords(node)

    rows = [
        f"<b>{html.escape(str(title))}</b>",
        f"<b>Graph node:</b> {node}",
        f"<b>Latitude:</b> {lat:.6f}",
        f"<b>Longitude:</b> {lon:.6f}"
    ]

    if details:

        for key, value in details.items():

            if value is None:
                continue

            if (
                isinstance(value, float)
                and np.isnan(value)
            ):
                continue

            rows.append(
                f"<b>{html.escape(str(key))}:</b> "
                f"{html.escape(str(value))}"
            )

    rows.append(
        (
            f'<a href="{google_maps_url(lat, lon)}" '
            f'target="_blank">'
            "Open in Google Maps"
            "</a>"
        )
    )

    return folium.Popup(
        "<br>".join(rows),
        max_width=420
    )


# ==================================================
# RIVER NETWORK
# ==================================================

for u, v, data in G.edges(
    data=True
):

    path_nodes = data.get(
        "original_path",
        [u, v]
    )

    line_coords = []

    # Use the detailed graph if it exists
    if G_detail is not None:

        for node in path_nodes:

            if node in G_detail.nodes:

                nd = (
                    G_detail.nodes[node]
                )

                line_coords.append(
                    [
                        float(
                            nd["latitude"]
                        ),
                        float(
                            nd["longitude"]
                        )
                    ]
                )

    # Fallback to straight compressed edge
    if len(line_coords) < 2:

        u_lat, u_lon = coords(u)
        v_lat, v_lon = coords(v)

        line_coords = [
            [
                u_lat,
                u_lon
            ],
            [
                v_lat,
                v_lon
            ]
        ]

    length_km = (
        float(
            data.get(
                "length_m",
                0
            )
        )
        / 1000
    )

    folium.PolyLine(
        line_coords,
        weight=3,
        opacity=0.65,
        color="#3388ff",
        tooltip=(
            f"{u} → {v} | "
            f"{length_km:.1f} km"
        )
    ).add_to(
        river_layer
    )


# ==================================================
# NETWORK SOURCES
# ==================================================

for node in source_nodes:

    lat, lon = coords(node)

    distance = (
        source_distance_lookup
        .get(
            node,
            np.nan
        )
    )

    # Source close to a measurement
    if pd.isna(distance):

        color = "red"

        coverage_text = (
            "No downstream measurement"
        )

    elif distance <= 100:

        color = "green"

        coverage_text = (
            f"{distance:.1f} km "
            "to first downstream measurement"
        )

    else:

        color = "orange"

        coverage_text = (
            f"{distance:.1f} km "
            "to first downstream measurement"
        )

    folium.Marker(
        [
            lat,
            lon
        ],

        tooltip=(
            f"Source {node}"
        ),

        popup=popup_html(
            node,
            "Network source",
            {
                "Coverage":
                    coverage_text,

                "In-degree":
                    G.in_degree(node),

                "Out-degree":
                    G.out_degree(node)
            }
        ),

        icon=folium.Icon(
            color=color,
            icon="play",
            prefix="fa"
        )

    ).add_to(
        source_layer
    )


# ==================================================
# OUTLET / DOWNSTREAM LEAF NODES
# ==================================================

for node in outlet_nodes:

    lat, lon = coords(node)

    folium.Marker(
        [
            lat,
            lon
        ],

        tooltip=(
            f"Outlet {node}"
        ),

        popup=popup_html(
            node,
            "Outlet / downstream leaf",
            {
                "In-degree":
                    G.in_degree(node),

                "Out-degree":
                    G.out_degree(node)
            }
        ),

        icon=folium.Icon(
            color="darkgreen",
            icon="stop",
            prefix="fa"
        )

    ).add_to(
        outlet_layer
    )


# ==================================================
# JUNCTION / MEETING POINTS
# ==================================================

for node in junction_nodes:

    lat, lon = coords(node)

    folium.CircleMarker(
        [
            lat,
            lon
        ],

        radius=7,

        color="#f59e0b",

        weight=2,

        fill=True,

        fill_color="#f59e0b",

        fill_opacity=0.75,

        tooltip=(
            f"Junction {node}"
        ),

        popup=popup_html(
            node,
            "Junction / meeting point",
            {
                "In-degree":
                    G.in_degree(node),

                "Out-degree":
                    G.out_degree(node)
            }
        )

    ).add_to(
        junction_layer
    )


# ==================================================
# WQ CLUSTER HELPER
# ==================================================

def member_station_text(
    node,
    role
):

    clusters = [
        c
        for c
        in G.nodes[node].get(
            "usgs_clusters",
            []
        )
        if c.get(
            "ml_role"
        ) == role
    ]

    members = sorted({
        sid

        for c in clusters

        for sid in c.get(
            "member_stations",
            []
        )
    })

    features = sorted({
        feature

        for c in clusters

        for feature in c.get(
            "features",
            []
        )
    })

    return (
        clusters,
        members,
        features
    )


# ==================================================
# TRAINING — 5/5
# ==================================================

for node in train_nodes:

    lat, lon = coords(node)

    (
        clusters,
        members,
        features
    ) = member_station_text(
        node,
        "train"
    )

    folium.Marker(
        [
            lat,
            lon
        ],

        tooltip=(
            "TRAIN: "
            +
            (
                " / ".join(
                    members
                )

                if members

                else str(node)
            )
        ),

        popup=popup_html(
            node,
            "Training — 5/5",
            {
                "Stations":
                    ", ".join(
                        members
                    ),

                "Features":
                    ", ".join(
                        features
                    )
            }
        ),

        icon=folium.Icon(
            color="green",
            icon="circle",
            prefix="fa"
        )

    ).add_to(
        train_layer
    )


# ==================================================
# HELD-OUT FULL TEST — 5/5
# ==================================================

for node in full_test_nodes:

    lat, lon = coords(node)

    (
        clusters,
        members,
        features
    ) = member_station_text(
        node,
        "test_full"
    )

    folium.Marker(
        [
            lat,
            lon
        ],

        tooltip=(
            "FULL TEST: "
            +
            (
                " / ".join(
                    members
                )

                if members

                else str(node)
            )
        ),

        popup=popup_html(
            node,
            "Held-out test — 5/5",
            {
                "Stations":
                    ", ".join(
                        members
                    ),

                "Features":
                    ", ".join(
                        features
                    )
            }
        ),

        icon=folium.Icon(
            color="red",
            icon="star",
            prefix="fa"
        )

    ).add_to(
        full_test_layer
    )


# ==================================================
# PARTIAL WQ TEST
# ==================================================

for node in partial_test_nodes:

    lat, lon = coords(node)

    (
        clusters,
        members,
        features
    ) = member_station_text(
        node,
        "test_partial"
    )

    feature_count = max(
        [
            c.get(
                "feature_count",
                0
            )

            for c in clusters
        ],
        default=0
    )

    missing = sorted(
        set([
            "DO concentration",
            "Water temperature",
            "Specific conductance",
            "Turbidity",
            "pH"
        ])
        -
        set(features)
    )

    folium.Marker(
        [
            lat,
            lon
        ],

        tooltip=(
            "PARTIAL TEST: "
            +
            (
                " / ".join(
                    members
                )

                if members

                else str(node)
            )
        ),

        popup=popup_html(
            node,
            "Partial WQ test",
            {
                "Stations":
                    ", ".join(
                        members
                    ),

                "Core features":
                    feature_count,

                "Available":
                    ", ".join(
                        features
                    ),

                "Missing":
                    ", ".join(
                        missing
                    )
            }
        ),

        icon=folium.Icon(
            color="orange",
            icon="flask",
            prefix="fa"
        )

    ).add_to(
        partial_test_layer
    )


# ==================================================
# HYDROLOGY SUPPORT
# ==================================================

for node in support_nodes:

    lat, lon = coords(node)

    gauges = (
        G.nodes[node]
        .get(
            "support_gauges",
            []
        )
    )

    gauge_ids = sorted({
        str(
            gauge.get(
                "station_id"
            )
        )

        for gauge in gauges

        if gauge.get(
            "station_id"
        ) is not None
    })

    used_by = sorted({
        station

        for gauge in gauges

        for station in gauge.get(
            "used_by_wq_stations",
            []
        )
    })

    folium.Marker(
        [
            lat,
            lon
        ],

        tooltip=(
            "HYDROLOGY: "
            +
            (
                " / ".join(
                    gauge_ids
                )

                if gauge_ids

                else str(node)
            )
        ),

        popup=popup_html(
            node,
            "Hydrology support",
            {
                "Gauge IDs":
                    ", ".join(
                        gauge_ids
                    ),

                "Used by WQ stations":
                    ", ".join(
                        used_by
                    )
            }
        ),

        icon=folium.Icon(
            color="purple",
            icon="tint",
            prefix="fa"
        )

    ).add_to(
        support_layer
    )


# ==================================================
# SEARCH INDEX
# ==================================================

search_features = []

for _, row in points_df.iterrows():

    search_name = (
        f"{row['type']} | "
        f"{row['label']} | "
        f"node {row['node_id']}"
    )

    search_features.append({
        "type":
            "Feature",

        "geometry": {
            "type":
                "Point",

            "coordinates": [
                float(
                    row[
                        "longitude"
                    ]
                ),
                float(
                    row[
                        "latitude"
                    ]
                )
            ]
        },

        "properties": {
            "search_name":
                search_name
        }
    })


search_layer = folium.GeoJson(
    {
        "type":
            "FeatureCollection",

        "features":
            search_features
    },

    name="Search index",

    marker=folium.CircleMarker(
        radius=1,
        opacity=0,
        fill_opacity=0
    ),

    show=False

).add_to(m)


plugins.Search(
    layer=search_layer,

    search_label="search_name",

    placeholder=(
        "Search station, node, "
        "source, junction..."
    ),

    collapsed=False,

    position="topleft",

    zoom=12

).add_to(m)


# ==================================================
# MAP CONTROLS
# ==================================================

plugins.Fullscreen(
    position="topleft",

    title="Fullscreen",

    title_cancel=(
        "Exit fullscreen"
    )

).add_to(m)


plugins.MiniMap(
    toggle_display=True,
    minimized=False

).add_to(m)


plugins.MousePosition(
    position="bottomright",

    separator=" | ",

    prefix="Lat/Lon:",

    num_digits=5

).add_to(m)


plugins.MeasureControl(
    position="topleft",

    primary_length_unit=
        "kilometers",

    secondary_length_unit=
        "miles"

).add_to(m)


folium.LayerControl(
    collapsed=False

).add_to(m)


# ==================================================
# LEGEND
# ==================================================

legend_html = """
<div style="
    position: fixed;
    bottom: 35px;
    left: 35px;
    z-index: 9999;
    background: rgba(255,255,255,0.94);
    border: 2px solid #777;
    border-radius: 8px;
    padding: 10px 12px;
    font-size: 13px;
    line-height: 1.45;
    box-shadow: 0 1px 6px rgba(0,0,0,0.25);
">

<b>River Network Legend</b>
<br>

<span style="color:#28a745;">
●
</span>
Training — 5/5
<br>

<span style="color:#d62728;">
★
</span>
Test — held-out 5/5
<br>

<span style="color:#f59e0b;">
◆
</span>
Test — partial WQ
<br>

<span style="color:#7e22ce;">
■
</span>
Hydrology support
<br>

<span style="color:#16a34a;">
▶
</span>
Source: ≤100 km to measurement
<br>

<span style="color:#f59e0b;">
▶
</span>
Source: >100 km to measurement
<br>

<span style="color:#dc2626;">
▶
</span>
Source: no downstream measurement
<br>

<span style="color:#15803d;">
■
</span>
Outlet / downstream leaf
<br>

<span style="color:#f59e0b;">
●
</span>
Junction / meeting point

</div>
"""


m.get_root().html.add_child(
    Element(
        legend_html
    )
)


# ==================================================
# FIT MAP TO FULL RIVER NETWORK
# ==================================================

m.fit_bounds(
    [
        [
            min(all_lat),
            min(all_lon)
        ],

        [
            max(all_lat),
            max(all_lon)
        ]
    ]
)


# ==================================================
# SAVE AND DISPLAY
# ==================================================

m.save(
    OUTPUT_HTML
)

print(
    "Saved interactive map:",
    OUTPUT_HTML
)

m

Saved interactive map: interactive_river_network_map.html


## 6. Optional quick summaries

In [9]:
if len(source_coverage):
    display(
        source_coverage.sort_values(
            "distance_to_first_measurement_km",
            ascending=False,
            na_position="first"
        )
    )

,source_node,nearest_measurement_node,distance_to_first_measurement_km
5,188990639,NaN,NaN
11,209256258,NaN,NaN
13,202613460,NaN,NaN
16,204021741,NaN,NaN
0,128398218,203177183.0,729.734489
18,171485821,151170041.0,711.542365
10,132116925,203177183.0,673.670218
1,105675404,131520862.0,370.704374
12,177526604,203177183.0,328.067555
14,64455269,67739434.0,159.991380


In [10]:
display(
    points_df.sort_values(
        ["type", "label"]
    )
)

,node_id,type,label,latitude,longitude,coverage,in_degree,out_degree,member_stations,feature_count,support_gauges
65,83102734,Hydrology support,USGS-09085100,39.555000,-107.337500,NaN,NaN,NaN,NaN,NaN,USGS-09085100
64,82757580,Hydrology support,USGS-09085500,39.574167,-107.446667,NaN,NaN,NaN,NaN,NaN,USGS-09085500
66,203417183,Hydrology support,USGS-09523000,32.870833,-114.480833,NaN,NaN,NaN,NaN,NaN,USGS-09523000
24,107686315,Junction / meeting point,Junction node 107686315,38.189167,-109.885833,NaN,2.0,1.0,NaN,NaN,NaN
25,113101070,Junction / meeting point,Junction node 113101070,37.888333,-110.390833,NaN,2.0,1.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
44,96496279,Training — 5/5,USGS-09180500,38.810833,-109.294167,NaN,NaN,NaN,USGS-09180500,NaN,NaN
38,67739434,Training — 5/5,USGS-09261000,40.408333,-109.234167,NaN,NaN,NaN,USGS-09261000,NaN,NaN
42,93300040,Training — 5/5,USGS-09315000,38.988333,-110.149167,NaN,NaN,NaN,USGS-09315000,NaN,NaN
46,126377584,Training — 5/5,USGS-09379500,37.150833,-109.866667,NaN,NaN,NaN,USGS-09379500,NaN,NaN


In [ ]:
import os
import shutil
import pickle
import pandas as pd
import networkx as nx
import importlib.metadata as metadata

# ============================================================
# SHARE PACKAGE SETTINGS
# ============================================================

SHARE_DIR = "river_network_lab_share"

folders = [
    SHARE_DIR,
    f"{SHARE_DIR}/graphs",
    f"{SHARE_DIR}/data",
    f"{SHARE_DIR}/notebooks",
    f"{SHARE_DIR}/exports",
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)


# ============================================================
# FILES TO COPY
# ============================================================

graph_files = [
    "master_compressed_river_graph.pkl",

    # Include this if you want them to have the original
    # detailed river paths too.
    "largest_river_component.pkl",
]

data_files = [
    "master_wq_station_table.csv",
    "usgs_cleaning_audit.csv",
    "usgs_all_stations_hourly.csv",
    "usgs_station_variable_completeness.csv",
]

notebook_files = [
    "Interactive_River_Network_Map.ipynb",
    "USGS_Data_Cleaning.ipynb",
]


def copy_if_exists(filename, destination):

    if os.path.exists(filename):

        shutil.copy2(
            filename,
            destination
        )

        print("Copied:", filename)

    else:
        print("Not found, skipped:", filename)


for filename in graph_files:
    copy_if_exists(
        filename,
        f"{SHARE_DIR}/graphs"
    )


for filename in data_files:
    copy_if_exists(
        filename,
        f"{SHARE_DIR}/data"
    )


for filename in notebook_files:
    copy_if_exists(
        filename,
        f"{SHARE_DIR}/notebooks"
    )


# ============================================================
# LOAD MASTER GRAPH
# ============================================================

with open(
    "master_compressed_river_graph.pkl",
    "rb"
) as f:

    G_share = pickle.load(f)


print()
print("Master graph:")
print("Nodes:", G_share.number_of_nodes())
print("Edges:", G_share.number_of_edges())
print("Directed:", G_share.is_directed())


# ============================================================
# PORTABLE NODE EXPORT
# ============================================================

node_rows = []

for node, data in G_share.nodes(data=True):

    node_rows.append({
        "node_id": node,

        "longitude":
            data.get("longitude"),

        "latitude":
            data.get("latitude"),

        "usgs_station_count":
            data.get(
                "usgs_station_count",
                0
            ),

        "support_gauge_count":
            data.get(
                "support_gauge_count",
                0
            ),

        "in_degree":
            G_share.in_degree(node),

        "out_degree":
            G_share.out_degree(node)
    })


nodes_df = pd.DataFrame(
    node_rows
)

nodes_df.to_csv(
    f"{SHARE_DIR}/exports/graph_nodes.csv",
    index=False
)


# ============================================================
# PORTABLE EDGE EXPORT
# ============================================================

edge_rows = []

for u, v, data in G_share.edges(data=True):

    edge_rows.append({
        "source": u,
        "target": v,

        "length_m":
            data.get("length_m"),

        "original_edge_count":
            data.get(
                "original_edge_count"
            ),

        "compressed_node_count":
            data.get(
                "compressed_node_count"
            )
    })


edges_df = pd.DataFrame(
    edge_rows
)

edges_df.to_csv(
    f"{SHARE_DIR}/exports/graph_edges.csv",
    index=False
)


print()
print(
    "Portable graph exports created."
)


# ============================================================
# REQUIREMENTS.TXT
# ============================================================

packages = [
    "networkx",
    "pandas",
    "numpy",
    "folium"
]

requirements = []

for package in packages:

    try:
        version = metadata.version(
            package
        )

        requirements.append(
            f"{package}=={version}"
        )

    except metadata.PackageNotFoundError:
        pass


with open(
    f"{SHARE_DIR}/requirements.txt",
    "w"
) as f:

    f.write(
        "\\n".join(
            requirements
        )
    )


# ============================================================
# README
# ============================================================

readme = """
COLORADO RIVER NETWORK PROJECT
==============================

Contents
--------

graphs/
    master_compressed_river_graph.pkl
        Main compressed NetworkX river graph used for analysis.

    largest_river_component.pkl
        Detailed river-network graph before compression.
        This file is optional but is useful for plotting the
        actual river geometry.

data/
    master_wq_station_table.csv
        Water-quality stations and associated metadata.

    usgs_cleaning_audit.csv
        Audit of downloaded/cleaned USGS series.

    usgs_all_stations_hourly.csv
        Combined cleaned hourly USGS observations.

    usgs_station_variable_completeness.csv
        Data completeness by station and variable.

exports/
    graph_nodes.csv
        Portable table of graph nodes.

    graph_edges.csv
        Portable table of directed graph edges.

notebooks/
    Interactive_River_Network_Map.ipynb
        Interactive Folium map.

    USGS_Data_Cleaning.ipynb
        USGS data-cleaning workflow.


LOADING THE GRAPH
-----------------

Python example:

    import pickle

    with open(
        "graphs/master_compressed_river_graph.pkl",
        "rb"
    ) as f:
        G = pickle.load(f)

    print(G.number_of_nodes())
    print(G.number_of_edges())
    print(G.is_directed())


IMPORTANT
---------

Only load pickle files from trusted sources.

The interactive-map notebook may request a CARTO
Basemaps API key. API keys are intentionally NOT
included in this package.

The graph is directed according to river flow.

Water-quality station information is stored primarily
in the "usgs_clusters" node attribute.

Hydrology-support gauges are stored in the
"support_gauges" node attribute.
"""


with open(
    f"{SHARE_DIR}/README.txt",
    "w"
) as f:

    f.write(readme)


# ============================================================
# ZIP EVERYTHING
# ============================================================

zip_path = shutil.make_archive(
    SHARE_DIR,
    "zip",
    SHARE_DIR
)


print()
print("====================================")
print("SHARE PACKAGE COMPLETE")
print("====================================")
print("Folder:", SHARE_DIR)
print("ZIP:", zip_path)